# Customer Review Analyzer — Review-Only Training
Only the review text is used as model input. Other fields (product, brand, user, date, location, source, IDs, etc.) are discarded. A sentiment label or rating is retained only as the training target.

## 1. Install and import

In [ ]:
!pip -q install scikit-learn
import os, re, json
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
SEED=42
np.random.seed(SEED); tf.random.set_seed(SEED)

## 2. Upload the three CSV datasets

In [ ]:
uploaded=files.upload()
def load_csv(path):
    for enc in ["utf-8-sig","cp1252","latin1"]:
        try:
            d=pd.read_csv(path,encoding=enc,sep=",",engine="python",on_bad_lines="warn")
            if len(d.columns)>1: return d
        except Exception: pass
    for enc in ["utf-8-sig","cp1252","latin1"]:
        try: return pd.read_csv(path,encoding=enc,sep=None,engine="python",on_bad_lines="warn")
        except Exception: pass
    raise ValueError("Could not read "+path)
datasets={}
for name in uploaded:
    datasets[name]=load_csv(name)
    print("\\n",name,"rows:",len(datasets[name]),"columns:",list(datasets[name].columns))

## 3. Keep only review text and sentiment/rating

In [ ]:
TEXT_CANDIDATES=["reviews.text","review_text","review","reviews","text","content","comment","customer_review"]
LABEL_CANDIDATES=["reviews.rating","sentiment","label","rating","score","stars","class","polarity"]
def find_col(df,candidates):
    cols={str(c).strip().lower():c for c in df.columns}
    return next((cols[x.lower()] for x in candidates if x.lower() in cols),None)
def to_sentiment(v):
    if pd.isna(v): return None
    s=str(v).strip().lower()
    try:
        n=float(s)
        if 1<=n<=2: return "negative"
        if n==3: return "neutral"
        if 4<=n<=5: return "positive"
    except ValueError: pass
    s=re.sub(r"[^a-z]","",s)
    return {"positive":"positive","pos":"positive","good":"positive",
            "negative":"negative","neg":"negative","bad":"negative",
            "neutral":"neutral","neu":"neutral","mixed":"neutral"}.get(s)

parts=[]
for name,df in datasets.items():
    tc,lc=find_col(df,TEXT_CANDIDATES),find_col(df,LABEL_CANDIDATES)
    print(name,"| review:",tc,"| sentiment/rating:",lc)
    if tc is None or lc is None:
        print("Skipped: missing review text or sentiment/rating target.")
        continue
    # Discard every other column. Only text and target remain.
    p=df[[tc,lc]].copy()
    p.columns=["review","raw_label"]
    p["review"]=p["review"].fillna("").astype(str).str.strip()
    p["sentiment"]=p["raw_label"].apply(to_sentiment)
    p=p[(p.review.str.len()>0)&p.sentiment.notna()]
    parts.append(p[["review","sentiment"]])

if not parts: raise ValueError("No usable review/label columns found. Check the printed column names.")
data=pd.concat(parts,ignore_index=True).drop_duplicates(["review","sentiment"])
print("\\nUsable reviews:",len(data))
print(data.sentiment.value_counts())
if set(data.sentiment.unique())!={"positive","negative","neutral"}:
    raise ValueError("Need all 3 classes. Review-only files without sentiment/rating labels cannot train a supervised classifier until labels are supplied.")
if data.sentiment.value_counts().min()<2: raise ValueError("Each class needs at least 2 reviews.")

## 4. Train the CNN model

In [ ]:
class_names=["negative","neutral","positive"]
label_map={v:i for i,v in enumerate(class_names)}
X=data.review.astype(str).values
y=np.array([label_map[v] for v in data.sentiment])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
vectorizer=tf.keras.layers.TextVectorization(max_tokens=20000,output_mode="int",output_sequence_length=120,standardize="lower_and_strip_punctuation")
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))
inputs=tf.keras.Input(shape=(),dtype=tf.string,name="review")
x=vectorizer(inputs)
x=tf.keras.layers.Embedding(20000,128)(x)
x=tf.keras.layers.Conv1D(128,5,activation="relu",padding="same")(x)
x=tf.keras.layers.GlobalMaxPooling1D()(x)
x=tf.keras.layers.Dropout(0.35)(x)
x=tf.keras.layers.Dense(64,activation="relu")(x)
x=tf.keras.layers.Dropout(0.25)(x)
outputs=tf.keras.layers.Dense(3,activation="softmax")(x)
model=tf.keras.Model(inputs,outputs)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
stop=tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=2,restore_best_weights=True)
history=model.fit(X_train,y_train,validation_split=0.15,epochs=10,batch_size=32,callbacks=[stop])

## 5. Evaluate and try a review

In [ ]:
loss,accuracy=model.evaluate(X_test,y_test,verbose=0)
print("Test accuracy:",round(float(accuracy),4))
pred=np.argmax(model.predict(X_test,verbose=0),axis=1)
print(classification_report(y_test,pred,labels=[0,1,2],target_names=class_names,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=[0,1,2],display_labels=class_names)
plt.show()
def analyze_review(review):
    probs=model.predict(np.array([review],dtype=object),verbose=0)[0]
    i=int(np.argmax(probs))
    return {"sentiment":class_names[i],"confidence":round(float(probs[i])*100,2),
            "scores":{class_names[j]:round(float(probs[j])*100,2) for j in range(3)}}
print(analyze_review("The product is excellent and delivery was very fast."))

## 6. Save and download the Keras model

In [ ]:
model.save("customer_review_analyzer.keras")
with open("sentiment_classes.json","w") as f: json.dump(class_names,f)
files.download("customer_review_analyzer.keras")
files.download("sentiment_classes.json")